In [1]:
import pandas as pd


In [4]:
df = pd.read_csv("merged_cricket_data.csv")

In [5]:
df["death_over"] = df["over"] >= 16

In [6]:
df[["over", "death_over"]].head(10)

,over,death_over
0,0,False
1,0,False
2,0,False
3,0,False
4,0,False
5,0,False
6,0,False
7,1,False
8,1,False
9,1,False


In [8]:
print(" its showing last 16-20 over which is death - over")

 its showing last 16-20 over which is death - over


In [9]:
print(" dot ball increase mental pressure , so create dot ball pressure")

 dot ball increase mental pressure , so create dot ball pressure


In [10]:
df["dot_ball"] = df["batsman_runs"] == 0


In [11]:
df[["batsman_runs", "dot_ball"]].head()

,batsman_runs,dot_ball
0,0,True
1,0,True
2,0,True
3,0,True
4,0,True


In [12]:
print("When a new batter comes, pressure increases.")

When a new batter comes, pressure increases.


In [13]:
df["new_batter"] = df.groupby("match_id")["batter"].shift(1) != df["batter"]

In [14]:
print(" If a wicket just fell → pressure increases.")

 If a wicket just fell → pressure increases.


In [15]:
df["recent_wicket"] = df["is_wicket"].shift(1) == 1

In [18]:
print("detect team collpase , team collpas happen wh nmany wickert fall")

detect team collpase , team collpas happen wh nmany wickert fall


In [23]:
df["ball_number"] = (df["over"] - 1) * 6 + df["ball"]

In [24]:
df["ball_number"] = df.groupby(["match_id", "inning"]).cumcount() + 1

In [25]:
df["recent_wickets"] = (
    df.groupby(["match_id", "inning"])["is_wicket"]
    .rolling(window=12, min_periods=1)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

In [26]:
df["team_collapse"] = df["recent_wickets"] >= 3

In [27]:
df[["recent_wickets", "team_collapse"]].head(20)

,recent_wickets,team_collapse
0,0.0,False
1,0.0,False
2,0.0,False
3,0.0,False
4,0.0,False
5,0.0,False
6,0.0,False
7,0.0,False
8,0.0,False
9,0.0,False


In [30]:
print("Traditional analysis defines collapse as many wickets lost overall.")

Traditional analysis defines collapse as many wickets lost overall.


In [31]:
df["current_score"] = df.groupby(["match_id", "inning"])["total_runs"].cumsum()


In [32]:
df["balls_bowled"] = (df["over"] - 1) * 6 + df["ball"]
df["balls_remaining"] = 120 - df["balls_bowled"]

In [33]:
df["required_runs"] = df["target_runs"] - df["current_score"]

In [34]:
df["required_rrr"] = df["required_runs"] / (df["balls_remaining"] / 6)

In [35]:
df["high_rrr"] = df["required_rrr"] > 10


In [36]:
df.columns


Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs',
       'total_runs', 'extras_type', 'is_wicket', 'player_dismissed',
       'dismissal_kind', 'fielder', 'id', 'season', 'city', 'date',
       'match_type', 'player_of_match', 'venue', 'team1', 'team2',
       'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin',
       'target_runs', 'target_overs', 'super_over', 'umpire1', 'umpire2',
       'death_over', 'dot_ball', 'new_batter', 'recent_wicket',
       'wickets_fallen', 'team_collapse', 'ball_number', 'recent_wickets',
       'current_score', 'balls_bowled', 'balls_remaining', 'required_runs',
       'required_rrr', 'high_rrr'],
      dtype='str')

In [37]:
df["pressure_score"] = (
    df["death_over"] * 1.3 +
    df["high_rrr"] * 1.1 +
    df["team_collapse"] * 1.0 +
    df["dot_ball"] * 0.8 +
    df["new_batter"] * 0.5 +
    df["recent_wicket"] * 0.5
)

In [38]:
df[["pressure_score"]].describe()

,pressure_score
count,260920.000000
mean,1.049829
std,0.870153
min,0.000000
25%,0.500000
50%,0.800000
75%,1.600000
max,5.200000


In [39]:
df["pressure_score"].value_counts().head(10)

pressure_score
0.0    54831
0.5    42026
1.3    39944
0.8    39576
1.1    13515
1.8    13183
1.6    11326
2.4    10670
1.9     7966
2.1     6083
Name: count, dtype: int64

In [40]:
df.shape
df.columns
df["pressure_score"].describe()


count    260920.000000
mean          1.049829
std           0.870153
min           0.000000
25%           0.500000
50%           0.800000
75%           1.600000
max           5.200000
Name: pressure_score, dtype: float64

In [41]:
df["pressure_score_norm"] = (
    df["pressure_score"] - df["pressure_score"].min()
) / (
    df["pressure_score"].max() - df["pressure_score"].min()
)



In [42]:
df["pressure_score_norm"].describe()

count    260920.000000
mean          0.201890
std           0.167337
min           0.000000
25%           0.096154
50%           0.153846
75%           0.307692
max           1.000000
Name: pressure_score_norm, dtype: float64

In [43]:
df[["pressure_score", "pressure_score_norm"]].head()

,pressure_score,pressure_score_norm
0,2.4,0.461538
1,2.4,0.461538
2,1.9,0.365385
3,1.9,0.365385
4,1.9,0.365385


In [44]:
df[[
    "pressure_score",
    "pressure_score_norm",
    "mental_breakdown"
]].head()

KeyError: "['mental_breakdown'] not in index"

In [45]:
df.columns

Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs',
       'total_runs', 'extras_type', 'is_wicket', 'player_dismissed',
       'dismissal_kind', 'fielder', 'id', 'season', 'city', 'date',
       'match_type', 'player_of_match', 'venue', 'team1', 'team2',
       'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin',
       'target_runs', 'target_overs', 'super_over', 'umpire1', 'umpire2',
       'death_over', 'dot_ball', 'new_batter', 'recent_wicket',
       'wickets_fallen', 'team_collapse', 'ball_number', 'recent_wickets',
       'current_score', 'balls_bowled', 'balls_remaining', 'required_runs',
       'required_rrr', 'high_rrr', 'pressure_score', 'pressure_score_norm'],
      dtype='str')

In [46]:
df["mental_breakdown"] = (
    (df["pressure_score_norm"] > 0.6) &
    ((df["is_wicket"] == 1) | (df["batsman_runs"] == 0))
).astype(int)

In [47]:
df["mental_breakdown"].value_counts()

mental_breakdown
0    255367
1      5553
Name: count, dtype: int64

In [48]:
df[[
    "pressure_score",
    "pressure_score_norm",
    "mental_breakdown"
]].head()

,pressure_score,pressure_score_norm,mental_breakdown
0,2.4,0.461538,0
1,2.4,0.461538,0
2,1.9,0.365385,0
3,1.9,0.365385,0
4,1.9,0.365385,0


In [49]:
df[["pressure_score", "pressure_score_norm"]].drop_duplicates().sort_values(by="pressure_score")


,pressure_score,pressure_score_norm
479,0.0,0.000000
28,0.5,0.096154
474,0.8,0.153846
497,1.0,0.192308
8,1.1,0.211538
119,1.3,0.250000
15767,1.5,0.288462
16,1.6,0.307692
118,1.8,0.346154
2,1.9,0.365385


In [50]:
df["mental_breakdown"].value_counts()


mental_breakdown
0    255367
1      5553
Name: count, dtype: int64

In [51]:
df[pressure_indicators].head()


NameError: name 'pressure_indicators' is not defined

In [52]:
df["match_type"].unique()


<StringArray>
[            'League',         'Semi Final',              'Final',
 '3rd Place Play-Off',        'Qualifier 1',  'Elimination Final',
        'Qualifier 2',         'Eliminator']
Length: 8, dtype: str

In [53]:
df["knockout_match"] = (df["match_type"] != "League").astype(int)

In [54]:
df[["match_type", "knockout_match"]].drop_duplicates()

,match_type,knockout_match
0,League,0
12801,Semi Final,1
13239,Final,1
41146,3rd Place Play-Off,1
57621,Qualifier 1,1
57870,Elimination Final,1
58111,Qualifier 2,1
93810,Eliminator,1


In [55]:
batsman_pressure_indicators = [
    "knockout_match",
    "death_over",
    "high_rrr",
    "team_collapse",
    "dot_ball",
    "new_batter",
    "recent_wicket"
]

In [56]:
df[batsman_pressure_indicators].head()

,knockout_match,death_over,high_rrr,team_collapse,dot_ball,new_batter,recent_wicket
0,0,False,True,False,True,True,False
1,0,False,True,False,True,True,False
2,0,False,True,False,True,False,False
3,0,False,True,False,True,False,False
4,0,False,True,False,True,False,False


In [57]:

df[df["knockout_match"] == 1][["match_type", "pressure_score", "pressure_score_norm"]].head(20)


,match_type,pressure_score,pressure_score_norm
12801,Semi Final,1.3,0.250000
12802,Semi Final,0.8,0.153846
12803,Semi Final,0.8,0.153846
12804,Semi Final,0.8,0.153846
12805,Semi Final,0.8,0.153846
12806,Semi Final,0.8,0.153846
12807,Semi Final,0.8,0.153846
12808,Semi Final,0.5,0.096154
12809,Semi Final,1.3,0.250000
12810,Semi Final,0.8,0.153846


In [58]:
df["pressure_score"] = (
    df["knockout_match"] * 1.5 +
    df["death_over"] * 1.3 +
    df["high_rrr"] * 1.1 +
    df["team_collapse"] * 1.0 +
    df["dot_ball"] * 0.8 +
    df["new_batter"] * 0.5 +
    df["recent_wicket"] * 0.5
)

In [59]:
df["pressure_score_norm"] = (
    (df["pressure_score"] - df["pressure_score"].min()) /
    (df["pressure_score"].max() - df["pressure_score"].min())
)

In [60]:
df[df["match_type"] == "Semi Final"][["pressure_score", "pressure_score_norm"]].head(20)

,pressure_score,pressure_score_norm
12801,2.8,0.417910
12802,2.3,0.343284
12803,2.3,0.343284
12804,2.3,0.343284
12805,2.3,0.343284
12806,2.3,0.343284
12807,2.3,0.343284
12808,2.0,0.298507
12809,2.8,0.417910
12810,2.3,0.343284


In [61]:
df["pressure_score"].max()

np.float64(6.7)

In [62]:
df["prev_over_runs"] = df.groupby(["match_id", "inning", "over"])["total_runs"].transform("sum").shift(1)
df["high_prev_over_runs"] = (df["prev_over_runs"] >= 12).astype(int)

In [63]:
df["low_rrr"] = (df["required_rrr"] <= 6).astype(int)

In [64]:
df["batter_balls_faced"] = df.groupby(["match_id", "inning", "batter"]).cumcount() + 1
df["set_batter"] = (df["batter_balls_faced"] >= 20).astype(int)

In [65]:
df["powerplay"] = (df["over"] <= 6).astype(int)

In [66]:
df["small_total"] = (df["target_runs"] <= 140).astype(int)

In [67]:
bowler_pressure_indicators = [
    "knockout_match",
    "death_over",
    "high_prev_over_runs",
    "low_rrr",
    "set_batter",
    "powerplay",
    "small_total"
]

In [68]:
df[bowler_pressure_indicators].head()

,knockout_match,death_over,high_prev_over_runs,low_rrr,set_batter,powerplay,small_total
0,0,False,0,0,0,1,0
1,0,False,0,0,0,1,0
2,0,False,0,0,0,1,0
3,0,False,0,0,0,1,0
4,0,False,0,0,0,1,0


In [69]:
df[bowler_pressure_indicators].sum().sort_values(ascending=False)

powerplay              95357
set_batter             72661
high_prev_over_runs    58097
low_rrr                50694
small_total            48795
death_over             46584
knockout_match         15822
dtype: int64

In [70]:
bowler_pressure_weights = {
    "knockout_match": 1.5,
    "death_over": 1.3,
    "high_prev_over_runs": 1.1,
    "low_rrr": 1.0,
    "set_batter": 0.8,
    "powerplay": 0.5,
    "small_total": 0.5
}

In [71]:
df["bowler_pressure_score"] = sum(
    df[col] * weight for col, weight in bowler_pressure_weights.items()
)

In [72]:
df["bowler_pressure_score_norm"] = (
    (df["bowler_pressure_score"] - df["bowler_pressure_score"].min()) /
    (df["bowler_pressure_score"].max() - df["bowler_pressure_score"].min())
)

In [73]:
df["bowler_pressure_score"].describe()


count    260920.000000
mean          1.261297
std           1.017296
min           0.000000
25%           0.500000
50%           1.000000
75%           2.000000
max           6.200000
Name: bowler_pressure_score, dtype: float64

In [74]:
df["bowler_pressure_score_norm"].describe()

count    260920.000000
mean          0.203435
std           0.164080
min           0.000000
25%           0.080645
50%           0.161290
75%           0.322581
max           1.000000
Name: bowler_pressure_score_norm, dtype: float64

In [75]:
df["bowler_breakdown"] = (
    (df["bowler_pressure_score_norm"] > 0.6) &
    (df["total_runs"] >= 4)
).astype(int)

In [76]:
df["bowler_breakdown"].value_counts()

bowler_breakdown
0    259227
1      1693
Name: count, dtype: int64

In [77]:
df["mental_breakdown"].value_counts()

mental_breakdown
0    255367
1      5553
Name: count, dtype: int64

In [78]:
df["bowler_breakdown"].value_counts()

bowler_breakdown
0    259227
1      1693
Name: count, dtype: int64

In [79]:
X_bat = df[batsman_pressure_indicators + ["pressure_score_norm"]]
y_bat = df["mental_breakdown"]


In [80]:
X_bowl = df[bowler_pressure_indicators + ["bowler_pressure_score_norm"]]
y_bowl = df["bowler_breakdown"]

In [81]:
X_bat.shape, X_bowl.shape

((260920, 8), (260920, 8))

In [82]:
df.groupby("pressure_score_norm")["mental_breakdown"].mean()

pressure_score_norm
0.000000    0.000000
0.074627    0.000000
0.119403    0.000000
0.149254    0.000000
0.164179    0.000000
0.194030    0.000000
0.223881    0.000000
0.238806    0.000000
0.268657    0.000000
0.283582    0.000000
0.298507    0.000000
0.313433    0.000000
0.343284    0.000000
0.358209    0.000000
0.373134    0.000000
0.388060    0.000000
0.417910    0.000000
0.432836    0.000000
0.447761    0.000000
0.462687    0.000000
0.462687    0.000000
0.477612    1.000000
0.492537    0.020388
0.507463    0.058206
0.522388    0.000000
0.537313    0.418605
0.537313    0.000000
0.552239    1.000000
0.567164    0.000000
0.582090    0.083660
0.611940    0.469636
0.626866    1.000000
0.641791    0.000000
0.656716    0.008547
0.686567    0.000000
0.701493    1.000000
0.716418    0.083333
0.731343    0.230769
0.761194    1.000000
0.776119    1.000000
0.805970    0.470588
0.835821    1.000000
0.850746    1.000000
0.880597    0.000000
0.925373    1.000000
1.000000    1.000000
Name: mental_b

In [83]:
df.groupby(df["pressure_score_norm"] > 0.6)["mental_breakdown"].mean()

pressure_score_norm
False    0.016308
True     0.637016
Name: mental_breakdown, dtype: float64

In [84]:
player_pressure = df.groupby("batter")["pressure_score_norm"].mean().sort_values(ascending=False)
player_pressure.head(10)

batter
U Kaul              0.776119
Sunny Gupta         0.731343
CRD Fernando        0.701493
KP Appanna          0.656716
PM Sarvesh Kumar    0.641791
Mohammad Asif       0.636816
KK Ahmed            0.635394
K Kartikeya         0.630310
Mayank Dagar        0.626866
RP Meredith         0.626866
Name: pressure_score_norm, dtype: float64

In [87]:
player_strength = df.groupby("batter")["mental_breakdown"].mean().sort_values()
player_strength.head(1000)

batter
JM Kemp           0.0
F Behardien       0.0
SD Hope           0.0
FH Edwards        0.0
GC Smith          0.0
                 ... 
Mayank Dagar      1.0
Abdur Razzak      1.0
V Pratap Singh    1.0
IC Pandey         1.0
S Kaushik         1.0
Name: mental_breakdown, Length: 673, dtype: float64

In [88]:
bowler_pressure = df.groupby("bowler")["bowler_pressure_score_norm"].mean().sort_values(ascending=False)
bowler_pressure.head(10)


bowler
LPC Silva      0.591398
YBK Jaiswal    0.548387
SPD Smith      0.516129
Sunny Gupta    0.508489
S Midhun       0.491315
D Brevis       0.473118
SN Khan        0.459677
ND Doshi       0.415393
B Chipli       0.411290
MJ Suthar      0.405914
Name: bowler_pressure_score_norm, dtype: float64

In [89]:
# Define high-pressure balls (you can adjust threshold)
high_pressure_balls_bat = df[df["pressure_score_norm"] > 0.6]

# Calculate Pressure Efficiency: runs scored relative to pressure faced
batter_pressure_efficiency = high_pressure_balls_bat.groupby("batter").apply(
    lambda x: x["batsman_runs"].sum() / x["pressure_score_norm"].sum()
)

# Sort to see top clutch batsmen
batter_pressure_efficiency = batter_pressure_efficiency.sort_values(ascending=False)

print("Top 10 Batsmen under Pressure:")
print(batter_pressure_efficiency.head(10))

Top 10 Batsmen under Pressure:
batter
JC Buttler      3.225926
DNT Zoysa       3.045455
L Wood          3.045455
DL Chahar       3.045455
Ishan Kishan    2.664773
RA Tripathi     2.310345
MJ Santner      1.905213
DH Yagnik       1.833333
LR Shukla       1.791444
IK Pathan       1.744792
dtype: float64


In [90]:
# Define high-pressure balls for bowlers
high_pressure_balls_bowl = df[df["bowler_pressure_score_norm"] > 0.6]

# Define scoring for each ball
def bowler_score(x):
    return (
        (x["is_wicket"] * 3) +          # Wicket = 3 points
        ((x["batsman_runs"] == 0).astype(int) * 1) -  # Dot ball = +1
        ((x["batsman_runs"] >= 4).astype(int) * 2)    # Boundary conceded = -2
    ).sum()

# Calculate Pressure Efficiency for each bowler
bowler_pressure_efficiency = high_pressure_balls_bowl.groupby("bowler").apply(
    lambda x: bowler_score(x) / x["bowler_pressure_score_norm"].sum()
)

# Sort to see top clutch bowlers
bowler_pressure_efficiency = bowler_pressure_efficiency.sort_values(ascending=False)

print("Top 10 Bowlers under Pressure:")
print(bowler_pressure_efficiency.head(10))

Top 10 Bowlers under Pressure:
bowler
D Brevis            5.904762
LE Plunkett         5.904762
M Ashwin            2.952381
JP Behrendorff      2.343643
R Sharma            2.039474
AA Chavan           1.589744
R Tewatia           1.589744
RE van der Merwe    1.557789
TP Sudhindra        1.476190
RS Bopara           1.476190
dtype: float64


In [91]:
# High pressure balls
high_pressure_balls_bat = df[df["pressure_score_norm"] > 0.6]

# Count high pressure balls per batter
pressure_ball_count = high_pressure_balls_bat.groupby("batter").size()

# Pressure Efficiency (runs / pressure faced)
pressure_efficiency = high_pressure_balls_bat.groupby("batter").apply(
    lambda x: x["batsman_runs"].sum() / x["pressure_score_norm"].sum()
)

# Clutch Index = efficiency * sqrt(high pressure balls)
batter_clutch_index = pressure_efficiency * pressure_ball_count.pow(0.5)

batter_clutch_index = batter_clutch_index.sort_values(ascending=False)

print("Top 10 Clutch Batters:")
print(batter_clutch_index.head(10))

Top 10 Clutch Batters:
batter
JC Buttler      7.901872
Ishan Kishan    7.537115
SS Tiwary       6.992790
DJ Bravo        6.958554
KA Pollard      6.508676
SL Malinga      6.349937
R Ashwin        6.337947
HH Pandya       5.679601
SK Raina        5.560038
DH Yagnik       5.500000
dtype: float64


In [92]:
# High pressure balls
high_pressure_balls_bowl = df[df["bowler_pressure_score_norm"] > 0.6]

# Count high pressure balls per bowler
pressure_ball_count_bowl = high_pressure_balls_bowl.groupby("bowler").size()

# Pressure Efficiency
def bowler_score(x):
    return (
        (x["is_wicket"] * 3) +          # Wicket = 3 points
        ((x["batsman_runs"] == 0).astype(int) * 1) -  # Dot ball = +1
        ((x["batsman_runs"] >= 4).astype(int) * 2)    # Boundary conceded = -2
    ).sum()

pressure_efficiency_bowl = high_pressure_balls_bowl.groupby("bowler").apply(
    lambda x: bowler_score(x) / x["bowler_pressure_score_norm"].sum()
)

# Clutch Index = efficiency * sqrt(high pressure balls)
bowler_clutch_index = pressure_efficiency_bowl * pressure_ball_count_bowl.pow(0.5)

bowler_clutch_index = bowler_clutch_index.sort_values(ascending=False)

print("Top 10 Clutch Bowlers:")
print(bowler_clutch_index.head(10))

Top 10 Clutch Bowlers:
bowler
JP Behrendorff     6.200696
D Brevis           5.904762
LE Plunkett        5.904762
JH Kallis          5.804711
YS Chahal          4.206230
M Ashwin           4.175297
Mohammad Asif      4.138348
R Sharma           4.078947
NM Coulter-Nile    4.052612
OC McCoy           3.993559
dtype: float64


In [93]:
# High pressure balls
high_pressure_balls_bat = df[df["pressure_score_norm"] > 0.6]

# Count high pressure balls per batter
pressure_ball_count = high_pressure_balls_bat.groupby("batter").size()

# Pressure Efficiency (runs / pressure faced)
pressure_efficiency = high_pressure_balls_bat.groupby("batter").apply(
    lambda x: x["batsman_runs"].sum() / x["pressure_score_norm"].sum()
)

# Clutch Index = efficiency * sqrt(high pressure balls)
batter_clutch_index = pressure_efficiency * pressure_ball_count.pow(0.5)

batter_clutch_index = batter_clutch_index.sort_values(ascending=False)

print("Top 10 Clutch Batters:")
print(batter_clutch_index.head(10))

Top 10 Clutch Batters:
batter
JC Buttler      7.901872
Ishan Kishan    7.537115
SS Tiwary       6.992790
DJ Bravo        6.958554
KA Pollard      6.508676
SL Malinga      6.349937
R Ashwin        6.337947
HH Pandya       5.679601
SK Raina        5.560038
DH Yagnik       5.500000
dtype: float64


In [94]:
# High pressure balls
high_pressure_balls_bowl = df[df["bowler_pressure_score_norm"] > 0.6]

# Count high pressure balls per bowler
pressure_ball_count_bowl = high_pressure_balls_bowl.groupby("bowler").size()

# Pressure Efficiency
def bowler_score(x):
    return (
        (x["is_wicket"] * 3) +          # Wicket = 3 points
        ((x["batsman_runs"] == 0).astype(int) * 1) -  # Dot ball = +1
        ((x["batsman_runs"] >= 4).astype(int) * 2)    # Boundary conceded = -2
    ).sum()

pressure_efficiency_bowl = high_pressure_balls_bowl.groupby("bowler").apply(
    lambda x: bowler_score(x) / x["bowler_pressure_score_norm"].sum()
)

# Clutch Index = efficiency * sqrt(high pressure balls)
bowler_clutch_index = pressure_efficiency_bowl * pressure_ball_count_bowl.pow(0.5)

bowler_clutch_index = bowler_clutch_index.sort_values(ascending=False)

print("Top 10 Clutch Bowlers:")
print(bowler_clutch_index.head(10))

Top 10 Clutch Bowlers:
bowler
JP Behrendorff     6.200696
D Brevis           5.904762
LE Plunkett        5.904762
JH Kallis          5.804711
YS Chahal          4.206230
M Ashwin           4.175297
Mohammad Asif      4.138348
R Sharma           4.078947
NM Coulter-Nile    4.052612
OC McCoy           3.993559
dtype: float64


In [95]:
# High-pressure balls
high_pressure_balls_bat = df[df["pressure_score_norm"] > 0.6]

# Count high-pressure balls per batter
pressure_ball_count = high_pressure_balls_bat.groupby("batter").size()

# Pressure Efficiency
pressure_efficiency = high_pressure_balls_bat.groupby("batter").apply(
    lambda x: x["batsman_runs"].sum() / x["pressure_score_norm"].sum()
)

# Consistency = #balls where batsman scored >0 / total high-pressure balls faced
consistency = high_pressure_balls_bat.groupby("batter").apply(
    lambda x: (x["batsman_runs"] > 0).sum() / len(x)
)

# Adjusted Clutch Index
batter_clutch_index = pressure_efficiency * pressure_ball_count.pow(0.5) * consistency
batter_clutch_index = batter_clutch_index.sort_values(ascending=False)

print("Top 10 Batsmen Clutch Index (with consistency):")
print(batter_clutch_index.head(10))

Top 10 Batsmen Clutch Index (with consistency):
batter
JC Buttler      5.267915
Ishan Kishan    4.710697
DH Yagnik       3.055556
DNT Zoysa       3.045455
DL Chahar       3.045455
L Wood          3.045455
SS Tiwary       2.719418
DJ Bravo        2.706104
BJ Hodge        2.636203
Ankit Sharma    2.463235
dtype: float64


In [96]:
# High-pressure balls for bowlers
high_pressure_balls_bowl = df[df["bowler_pressure_score_norm"] > 0.6]

# Count high-pressure balls
pressure_ball_count_bowl = high_pressure_balls_bowl.groupby("bowler").size()

# Pressure Efficiency
def bowler_score(x):
    return (
        (x["is_wicket"] * 3) +          
        ((x["batsman_runs"] == 0).astype(int) * 1) -  
        ((x["batsman_runs"] >= 4).astype(int) * 2)    
    ).sum()

pressure_efficiency_bowl = high_pressure_balls_bowl.groupby("bowler").apply(
    lambda x: bowler_score(x) / x["bowler_pressure_score_norm"].sum()
)

# Consistency = #balls with wickets or dot balls / total high-pressure balls
consistency_bowl = high_pressure_balls_bowl.groupby("bowler").apply(
    lambda x: (((x["is_wicket"]==1) | (x["batsman_runs"]==0))).sum() / len(x)
)

# Adjusted Clutch Index
bowler_clutch_index = pressure_efficiency_bowl * pressure_ball_count_bowl.pow(0.5) * consistency_bowl
bowler_clutch_index = bowler_clutch_index.sort_values(ascending=False)

print("Top 10 Bowlers Clutch Index (with consistency):")
print(bowler_clutch_index.head(10))

Top 10 Bowlers Clutch Index (with consistency):
bowler
D Brevis           5.904762
LE Plunkett        5.904762
JP Behrendorff     4.429068
JH Kallis          2.579871
ND Doshi           2.159443
M Ashwin           2.087649
R Sharma           2.039474
OC McCoy           1.996779
RJ Harris          1.776020
NM Coulter-Nile    1.642951
dtype: float64
